In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Forecast')

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import utils.plot_dashes as plot_dashes

from conf.processing import DataProcessing
from utils import genetic_algorithm as gen_alg
from models import ann_models, kernel_method, tree_based_ensemble as tree_based
from sklearn.metrics import mean_squared_error

pd.set_option('display.float_format', '{:.6f}'.format)

Dataset

In [ ]:
all_results = []
all_runs = []
ga_runs = []

def get_file():

    file_path = '/content/drive/MyDrive/Colab Notebooks/Forecast/Datasets'

    for f in os.listdir(file_path):
        full_path = os.path.join(file_path, f)

        yield full_path

for f in get_file():

    dataset = pd.read_csv(f, sep='\t', decimal='.')
    raw_data = DataProcessing(dataset, n_lags=5)
    prep_data = raw_data.prepare_data()

    mlp = ann_models.MLP()
    results_MLP = mlp.train(prep_data)
    MLP_forecast_v = np.array(results_MLP['pred_valid'])
    MLP_forecast = np.array(results_MLP['pred_test_denom'])
    
    scn = ann_models.GridSCN()
    results_SCN = scn.train(prep_data)
    SCN_forecast_v = np.array(results_SCN['pred_valid'])
    SCN_forecast = np.array(results_SCN['pred_test'])

    elm = ann_models.GridELM()
    results_ELM = elm.train(prep_data)
    ELM_forecast_v = np.array(results_ELM['pred_valid'])
    ELM_forecast = np.array(results_ELM['pred_test'])

    svr = kernel_method.SVM()
    results_SVR = svr.train(prep_data)
    SVR_forecast_v = np.array(results_SVR['pred_valid'])
    SVR_forecast = np.array(results_SVR['pred_test'])

    rf = tree_based.RF()
    results_RF = rf.train(prep_data)
    RF_forecast_v = np.array(results_RF['pred_valid'])
    RF_forecast = np.array(results_RF['pred_test'])

    gb = tree_based.GBoosting()
    results_GB = gb.train(prep_data)
    GB_forecast_v = np.array(results_GB['pred_valid'])
    GB_forecast = np.array(results_GB['pred_test'])

    P_v = np.column_stack(
        (
            MLP_forecast_v, 
            SCN_forecast_v,
            ELM_forecast_v, 
            SVR_forecast_v, 
            RF_forecast_v, 
            GB_forecast_v
        )
    )

    P = np.column_stack(
        (
            MLP_forecast, 
            SCN_forecast,
            ELM_forecast, 
            SVR_forecast, 
            RF_forecast, 
            GB_forecast
        )
    )
    
    for x in range(30):
        
        ga = gen_alg.GeneticAlgorithm()
        
        ga_result = ga.execute(P_v, prep_data['target_valid'])

        y_hat = P @ ga_result['best_ind']
        ga_metrics = plot_dashes.get_metrics_error(prep_data['target_test'], y_hat)
        ga_runs.append({

            'Dataset': os.path.basename(f),
            'Run': x,

            'MSE': ga_metrics['MSE'],
            'RMSE': ga_metrics['RMSE'],
            'MAE': ga_metrics['MAE'],
            'MAPE': ga_metrics['MAPE'],

            'Weights': ga_result['best_ind'],
            'Fitness': ga_result['best_fitness']

        })
    
    
    ensemble_mean = np.mean(P, axis=1)

    MLP = plot_dashes.get_metrics_error(prep_data['target_test'], MLP_forecast)
    SCN = plot_dashes.get_metrics_error(prep_data['target_test'], SCN_forecast)
    ELM = plot_dashes.get_metrics_error(prep_data['target_test'], ELM_forecast)
    SVR = plot_dashes.get_metrics_error(prep_data['target_test'], SVR_forecast)
    RF = plot_dashes.get_metrics_error(prep_data['target_test'], RF_forecast)
    GB = plot_dashes.get_metrics_error(prep_data['target_test'], GB_forecast)
    EMEAN = plot_dashes.get_metrics_error(prep_data['target_test'], ensemble_mean)
    GA = plot_dashes.get_metrics_error(prep_data['target_test'], y_hat)

    all_results.append({
        'dataset': os.path.basename(f),
        'MLP': MLP,
        'SCN': SCN,
        'ELM': ELM,
        'SVR': SVR,
        'RF': RF,
        'GB': GB,
        'EMEAN': EMEAN,
        'GA': GA
    })

    all_runs.append({
        'dataset': os.path.basename(f),
        'model': 'MLP',
        'runs': results_MLP['lst_results']
    })

    all_runs.append({
        'dataset': os.path.basename(f),
        'model': 'SCN',
        'runs': results_SCN['lst_results']
    })

    all_runs.append({
        'dataset': os.path.basename(f),
        'model': 'ELM',
        'runs': results_ELM['lst_results']
    })

    all_runs.append({
        'dataset': os.path.basename(f),
        'model': 'SVR',
        'runs': results_SVR['lst_results']
    })

    all_runs.append({
        'dataset': os.path.basename(f),
        'model': 'RF',
        'runs': results_RF['lst_results']
    })

    all_runs.append({
        'dataset': os.path.basename(f),
        'model': 'GB',
        'runs': results_GB['lst_results']
    })
    

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
df_results = pd.DataFrame(all_results)
df_runs = pd.DataFrame(all_runs)

In [ ]:
# Tabela Principal (MSE, RMSE, MAE e MAPE)
rows = []

for _, row in df_results.iterrows():

    dataset = row['dataset']

    for model in ['MLP','SCN','ELM','SVR','RF','GB', 'EMEAN', 'GA']:
        metrics = row[model]
        rows.append({

            'Dataset': dataset,
            'Model': model,

            'MSE': metrics['MSE'],
            'RMSE': metrics['RMSE'],
            'MAE': metrics['MAE'],
            'MAPE': metrics['MAPE']

        })

df_metrics = pd.DataFrame(rows)

In [16]:
df_metrics

,Dataset,Model,MSE,RMSE,MAE,MAPE
0,airlines.txt,MLP,3.677686e+03,60.643924,46.947457,0.099546
1,airlines.txt,SCN,5.813661e+03,76.247369,61.468551,0.129101
2,airlines.txt,ELM,1.991447e+05,446.256275,439.380831,0.997860
3,airlines.txt,SVR,1.522242e+04,123.379190,105.387674,0.222434
4,airlines.txt,RF,2.215430e+04,148.843192,127.779707,0.269653
5,airlines.txt,GB,2.015455e+04,141.966732,120.217282,0.252672
6,airlines.txt,GA,2.237302e+04,149.576151,138.574087,0.303453
7,coloradoRiver.txt,MLP,2.978913e-01,0.545794,0.407296,1.312045
8,coloradoRiver.txt,SCN,3.744613e-01,0.611932,0.413207,1.131296
9,coloradoRiver.txt,ELM,9.289270e-01,0.963809,0.762711,0.923693


In [ ]:
# Média das Métricas por Modelo
df_metrics.groupby('Model')[['MSE','RMSE','MAE','MAPE']].mean()

,MSE,RMSE,MAE,MAPE
Model,,,,
ELM,1.237984e+06,659.748196,546.196690,0.969744
GA,1.841328e+05,249.378662,191.776059,0.445161
GB,2.130959e+05,264.281201,216.764766,0.676636
MLP,1.391845e+05,201.782364,152.773564,0.592510
RF,1.637547e+05,236.826305,182.850561,0.433501
SCN,1.547411e+05,215.536209,152.217582,0.635340
SVR,4.077809e+05,349.343362,264.713658,0.451048


In [ ]:
# Ranking dos Modelos
ranking = []

for dataset in df_metrics['Dataset'].unique():

    temp = df_metrics[df_metrics['Dataset']==dataset]
    temp = temp.sort_values('MSE')
    temp['Rank'] = range(1,len(temp)+1)
    ranking.append(temp)

df_rank = pd.concat(ranking)

In [ ]:
# Média e Desvio das Execuções

rows=[]

for _,row in df_runs.iterrows():

    for i,mse in enumerate(row['runs']):

        rows.append({

            'Dataset':row['dataset'],
            'Model':row['model'],
            'Run':i,
            'MSE':mse

        })

df_runs_expand = pd.DataFrame(rows)

df_runs_expand.groupby(
    ['Dataset','Model']
)['MSE'].agg(
    ['mean','std','min','max']
)

In [ ]:
# Organizar o GA

ga_runs.append({

    'Dataset':os.path.basename(f),

    'Run':x,

    'MSE':metrics['MSE'],

    'RMSE':metrics['RMSE'],

    'MAE':metrics['MAE'],

    'MAPE':metrics['MAPE'],

    'Weights':ga_result['best_ind'],

    'Fitness':ga_result['best_fitness']

})

df_ga = pd.DataFrame(ga_runs)
df_ga.groupby('Dataset')['MSE'].mean()
df_ga.groupby('Dataset')['MSE'].std()

In [ ]:
# Curva de Convergência

ga_result['fitness_curve']
plt.plot(ga_result['fitness_curve'])

plt.xlabel("Generation")

plt.ylabel("Fitness")

plt.grid()

plt.show()

In [ ]:
# Friedman

friedman_df = df_metrics.pivot(

    index='Dataset',

    columns='Model',

    values='MSE'

)

from scipy.stats import friedmanchisquare

friedmanchisquare(

    friedman_df['MLP'],
    friedman_df['SCN'],
    friedman_df['ELM'],
    friedman_df['SVR'],
    friedman_df['RF'],
    friedman_df['GB'],
    friedman_df['GA']

)

In [ ]:
pip install scikit-posthocs

In [ ]:
import scikit_posthocs as sp

nemenyi = sp.posthoc_nemenyi_friedman(

    friedman_df.values

)

print(nemenyi)